## Worker-Object 模式优雅退出的三步走

当前代码中的实现就是标准做法：

### 第 1 步：取消 worker 的耗时操作

```python
# CountingWorker 中
def run(self) -> None:
    for value in range(self.limit + 1):
        if self._cancel_requested.is_set():   # ← 检查取消标志
            self.finished.emit("任务已取消")
            return                            # ← 立即退出循环

def cancel(self) -> None:
    self._cancel_requested.set()              # 设置取消标志
```

关键是 **worker 内部定期检查取消标志**，不能死循环里一直算。

### 第 2 步：信号链自动触发线程退出

```python
# start_work() 中已连好的信号链
self.worker.finished.connect(self.thread.quit)    # worker 完成 → 线程退出
self.worker.failed.connect(self.thread.quit)      # worker 失败 → 线程退出
```

worker 发射 `finished` 信号 → `thread.quit()` 被调用 → 线程的事件循环退出 → `thread.finished` 信号发射 → `deleteLater` 清理。

### 第 3 步：等待确认（关闭窗口时）

```python
# closeEvent 中
self.worker.cancel()                    # ① 设标志让 worker 自己停
self.thread.quit()                      # ② 退出事件循环
if not self.thread.wait(3000):          # ③ 等最多 3 秒
    self.thread.terminate()             # ④ 超时强制终止（兜底）
    self.thread.wait()
```

### 完整退出流程

```
cancel() 设标志
  → worker.run() 检测到标志，finished.emit()
    → thread.quit()  事件循环退出
      → thread.finished 信号
        → thread.deleteLater()  标记删除
        → clear_thread_refs()   清理引用

若 3 秒内没走完 → terminate() 暴力兜底
```

### 优雅退出的关键点

| 要点 | 说明 |
|---|---|
| **cancel 是请求，不是强制** | `cancel()` 只设标志，worker 自己检查并退出 |
| **不要直接 terminate** | `terminate()` 是兜底手段，正常情况下走信号链 |
| **wait 是同步等待** | 等线程真正结束再继续，否则窗口析构时可能崩溃 |
| **cleanup 靠信号链** | `deleteLater` + `clear_thread_refs`，不手动 `del` |

### 信号链是自动的，你只需调 cancel

正常运行时用户点"取消"按钮，只需调 `worker.cancel()`，后面的事信号链自动完成：

```
用户点取消 → cancel() → worker.run 检测 → finished.emit → quit → finished → deleteLater
```

只有**关闭窗口**这种紧急情况才需要 `wait` + `terminate` 兜底。